# Module 02: Pandas for Machine Learning
## Notebook 03: Data Cleaning and Missing Value Imputation

Real-world datasets are rarely clean. Machine learning algorithms (with rare exceptions like LightGBM or XGBoost) cannot handle missing values (`NaN`, `None`) or noisy inconsistent string formatting. Rigorous, leak-free cleaning is an essential prerequisite for model training.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Detect and quantify missing values across columns and rows.
2. Select between dropping samples and statistical imputation.
3. Impute numerical features using mean, median, and categorical features using mode.
4. Detect and reconcile duplicate records.
5. Clean noisy string features using vectorized string methods (`.str` accessor).
6. **Advanced:** Implement subgroup-conditional imputation, missingness indicators, and structured regex parsing pipelines.

In [ ]:
import pandas as pd
import numpy as np

# Create a dirty synthetic dataset typical of raw database extracts
dirty_data = {
    'Applicant_ID': [101, 102, 103, 104, 105, 106, 106, 107, 108],
    'Name': [' Alice Smith ', 'Bob J.', ' Charlie Brown', 'Diana Prince', 'Evan Wright ', 'Fiona Gallagher', 'Fiona Gallagher', ' George Clark', 'Hannah Abbott'],
    'Department': ['Engineering', 'Sales', 'Engineering', np.nan, 'Finance', 'Engineering', 'Engineering', 'Sales', 'Finance'],
    'Income': [85000.0, 54000.0, np.nan, 92000.0, 68000.0, 115000.0, 115000.0, np.nan, 72000.0],
    'Credit_Score': [720, np.nan, 680, 750, 610, 810, 810, 590, 705],
    'Contact_Info': ['alice@company.com (ext 12)', 'bob@sales.org', 'charlie@company.com (ext 99)', 'diana@corp.net', 'evan@finance.org', 'fiona@tech.io', 'fiona@tech.io', 'george@sales.org', 'hannah@finance.org']
}

df_raw = pd.DataFrame(dirty_data)
print("Raw Dirty Dataset:")
print(df_raw)

---
### 1. Detecting and Quantifying Missing Values

Pandas represents missing data with `np.nan` (or `pd.NA`).
Key diagnostic methods:
- `df.isna()` / `df.isnull()`: Boolean mask of missing values.
- `df.isna().sum()`: Count of missing entries per feature column.
- `df.isna().mean() * 100`: Percentage of missing data per feature.

In [ ]:
print("Missing Values Count per Column:")
print(df_raw.isna().sum())

print("\nMissing Values Percentage:")
missing_pct = (df_raw.isna().mean() * 100).round(1)
print(missing_pct)

---
### 2. Dropping vs. Imputing Missing Values

- **Dropping rows (`dropna(axis=0)`):** Acceptable if the missingness percentage is extremely small (< 2%) and random.
- **Dropping columns (`dropna(axis=1)`):** Common if a feature is missing > 50-70% of its data and cannot be imputed.
- In most ML pipelines, dropping data loses valuable statistical power; **imputation** is preferred.

In [ ]:
# Dropping rows with ANY missing value
df_dropped = df_raw.dropna(axis=0)
print(f"Original rows: {len(df_raw)} | Rows after dropna: {len(df_dropped)}")

# Dropping rows only if specific key columns are missing
df_clean_key = df_raw.dropna(subset=['Applicant_ID', 'Income'])
print(f"Rows after dropping missing Income: {len(df_clean_key)}")

---
### 3. Statistical Imputation Strategies

Standard univariate imputation approaches:
- **Numerical Features:** Impute with the **median** (robust to skewed outliers) or **mean**.
- **Categorical Features:** Impute with the **mode** (most frequent class) or a dedicated `'Missing'` token.
- **Missingness Indicator:** Creating a binary flag column indicating whether the original value was imputed preserves the informative signal that data was missing.

In [ ]:
df_imputed = df_raw.copy()

# Add missingness indicator before filling
df_imputed['Income_Was_Missing'] = df_imputed['Income'].isna().astype(int)

# Impute numerical feature with median
income_median = df_imputed['Income'].median()
df_imputed['Income'] = df_imputed['Income'].fillna(income_median)

# Impute categorical feature with mode
dept_mode = df_imputed['Department'].mode()[0]
df_imputed['Department'] = df_imputed['Department'].fillna(dept_mode)

print(f"Imputed Income with Median: ${income_median:,.2f}")
print(f"Imputed Department with Mode: '{dept_mode}'")
print("\nImputed DataFrame Preview:")
print(df_imputed[['Applicant_ID', 'Department', 'Income', 'Income_Was_Missing']])

---
### 4. Detecting and Removing Duplicates

Duplicate records falsely inflate the importance of particular observations and distort cross-validation.
- `df.duplicated()`: Returns boolean mask of duplicate rows.
- `df.drop_duplicates()`: Removes duplicate observations.

In [ ]:
# Check for exact duplicate rows
duplicate_mask = df_imputed.duplicated(subset=['Applicant_ID'], keep='first')
print("Duplicate rows detected:\n", df_imputed[duplicate_mask])

# Drop duplicates in-place or returning new df
df_deduped = df_imputed.drop_duplicates(subset=['Applicant_ID'], keep='first').copy()
print(f"\nRows before deduplication: {len(df_imputed)} | After: {len(df_deduped)}")

---
### 5. Cleaning String Features with `.str`

Text features frequently contain leading/trailing whitespaces, irregular capitalization, or extraneous symbols.
The `.str` accessor provides vectorized string transformations:
- `.str.strip()`: Remove surrounding whitespace.
- `.str.lower()` / `.str.upper()`: Standardize casing.
- `.str.replace()`: Substitute characters or patterns.

In [ ]:
# Clean name column
df_deduped['Name'] = df_deduped['Name'].str.strip().str.title()

# Clean domain from contact email
df_deduped['Email_Domain'] = df_deduped['Contact_Info'].str.extract(r'@([a-zA-Z0-9.-]+)')

print("Cleaned String Features:")
print(df_deduped[['Name', 'Email_Domain']])

---
### 6. Advanced Complex Usage: Subgroup-Conditional Imputation and Structured Regex Pipelines

A major flaw of global univariate imputation is **group bias**:
- Imputing missing Income with the global dataset median replaces an Engineering salary with an aggregate figure skewed by lower-paying departments.
- **Subgroup-Conditional Imputation** computes the median/mode within each category (`groupby().transform()`), drastically reducing estimation error.

Furthermore, real-world data science involves extracting multiple structured fields simultaneously from complex unstructured strings (e.g., server logs, addresses, diagnostic codes) using named regex groups.

In [ ]:
# Advanced Demonstration: Subgroup-Conditional Imputation
# Create dataset where income differs dramatically by department and experience tier
cohort_data = pd.DataFrame({
    'Dept': ['Tech', 'Tech', 'Tech', 'Sales', 'Sales', 'Sales', 'HR', 'HR', 'HR'],
    'Tier': ['Senior', 'Junior', 'Senior', 'Senior', 'Junior', 'Junior', 'Senior', 'Junior', 'Junior'],
    'Salary': [140000.0, 85000.0, np.nan, 95000.0, 50000.0, np.nan, 75000.0, np.nan, 45000.0]
})

print("Original Cohort Data with Missing Salaries:")
print(cohort_data)

# Impute salary conditionally by [Dept, Tier] subgroup median
# Notice the clean transform syntax:
cohort_data['Salary_Imputed_Conditional'] = cohort_data.groupby(['Dept', 'Tier'])['Salary'].transform(
    lambda group: group.fillna(group.median())
)

# Contrast with naive global median
cohort_data['Salary_Imputed_Global'] = cohort_data['Salary'].fillna(cohort_data['Salary'].median())

print("\nComparison of Group-Conditional vs. Naive Global Imputation:")
print(cohort_data[['Dept', 'Tier', 'Salary', 'Salary_Imputed_Conditional', 'Salary_Imputed_Global']])

# Advanced Regex Feature Extraction with Named Capture Groups
complex_logs = pd.Series([
    "ID:US-901; STATUS:OK; LATENCY:42ms; IP:192.168.1.1",
    "ID:EU-442; STATUS:FAIL; LATENCY:312ms; IP:10.0.0.45",
    "ID:AP-108; STATUS:OK; LATENCY:18ms; IP:172.16.254.1"
], name="log_string")

pattern = r"ID:(?P<Region>[A-Z]{2})-(?P<Device_ID>\d+); STATUS:(?P<Status>\w+); LATENCY:(?P<Latency_ms>\d+)ms; IP:(?P<IP_Address>[\d.]+)"
extracted_features = complex_logs.str.extract(pattern)
extracted_features['Latency_ms'] = pd.to_numeric(extracted_features['Latency_ms'])

print("\nStructured Features Extracted via Named Regex Groups:")
print(extracted_features)

### Summary & Next Steps
In this notebook, you mastered:
- Diagnosing missingness distributions and percentages.
- Safe statistical imputation (mean, median, mode) and missingness indicator flags.
- Removing duplicate records and applying vectorized string cleaning.
- Subgroup-conditional imputation (`groupby().transform()`) to eliminate category bias.
- Multi-field feature extraction from unstructured strings with named regex groups.

**Next Notebook:** `04_aggregations_grouping_and_pivot_tables.ipynb` — Master Split-Apply-Combine, multi-metric aggregations, and pivot tables.